In [1]:
!pip install -q pytesseract pdf2image pillow opencv-python anthropic
!apt-get install -q poppler-utils
!apt-get install -q tesseract-ocr-ara

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr-ara is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
#claude_model = "claude-sonnet-4-20250514"
claude_model = "claude-haiku-4-5-20251001"

In [3]:
import os

output_dir = 'ara-ocr'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
print(f"Created directory: {output_dir}")

Created directory: ara-ocr


In [4]:
import cv2
import numpy as np
import pytesseract
import re
from pdf2image import convert_from_path
from PIL import Image

def optimize_pdf_ocr(pdf_path, lang='ara'):
    pages = convert_from_path(pdf_path, 300)
    full_text = []
    for i, page in enumerate(pages):
        open_cv_image = np.array(page)
        image_rgb = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)
        image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
        text = pytesseract.image_to_string(image_rgb, lang=lang)
        full_text.append(f"--- Page {i+1} ---\n{text}")
    return "\n".join(full_text)

pdf_file = 'pages_5_7.pdf'
optimized_result = optimize_pdf_ocr(pdf_file)

with open(os.path.join(output_dir, 'optimized_approach.txt'), 'w', encoding='utf-8') as f:
    f.write(optimized_result)
print('OCR complete.')

OCR complete.


In [5]:
import anthropic
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def refine_with_llm(raw_text, model_name=claude_model):
    prompt = f"""
أنت خبير في تحقيق النصوص التاريخية. المهمة هي معالجة نص OCR من مذكرات جعفر العسكري.

المطلوب تنفيذ القواعد التالية بدقة:
1. دمج الأسطر (Flowing Text): قم بإلغاء الفواصل المصطنعة بين الأسطر الناتجة عن التنسيق العمودي للكتاب، واجعل النص يتدفق كفقرات متصلة وطبيعية.
2. إصلاح الأخطاء فقط: صحح الكلمات المشوهة آليّاً دون تغيير الأسلوب أو إضافة كلمات.
3. حافظ على الأسماء التاريخية والجغرافية كما وردت.
4. لا تضع نقطة نهاية إذا انتهى النص بجملة مفتوحة.
5. العناوين والأسماء الرئيسية تبقى في أسطر مستقلة.
6. أخرِج النص المصحح مباشرةً فقط، بدون أي عنوان أو تعليق في البداية.

النص الخام:
{raw_text}
    """
    try:
        message = client.messages.create(
            model=model_name,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    except Exception as e:
        print(f"  Warning: {e}")
        return raw_text

# Process page by page
page_blocks = re.split(r'(?=--- Page \d+ ---)', optimized_result)
corrected_pages = []
for block in page_blocks:
    block = block.strip()
    if not block:
        continue
    m = re.match(r'(--- Page \d+ ---)\n?(.*)', block, re.DOTALL)
    if m:
        header, page_text = m.group(1), m.group(2).strip()
        if page_text:
            print(f"Correcting {header}...")
            corrected = refine_with_llm(page_text)
            corrected_pages.append(f"{header}\n{corrected}")
        else:
            corrected_pages.append(block)
    else:
        corrected_pages.append(block)

clean_text = "\n\n".join(corrected_pages)
with open(os.path.join(output_dir, 'LLM_corrected_text.txt'), 'w', encoding='utf-8') as f:
    f.write(clean_text)
print('Per-page correction done. Saved LLM_corrected_text.txt')

Correcting --- Page 1 ---...
Correcting --- Page 2 ---...
Correcting --- Page 3 ---...
Per-page correction done. Saved LLM_corrected_text.txt


In [6]:
import json

def process_boundary(prev_content, next_content, page_num, model_name=claude_model):
    """
    For the boundary between page_num and page_num+1:
      1. Detect and strip the running header/footer at the top of next_content.
      2. If the last sentence of prev_content is incomplete, join it with the
         first content line of next_content.
    Returns (updated_prev_content, updated_next_content).
    """
    prev_lines = [l for l in prev_content.splitlines() if l.strip()]
    next_lines = [l for l in next_content.splitlines() if l.strip()]
    if not prev_lines or not next_lines:
        return prev_content, next_content

    prev_tail = '\n'.join(prev_lines[-2:])
    next_head = '\n'.join(next_lines[:3])

    prompt = (
        f"نهاية الصفحة {page_num}:\n{prev_tail}\n\n"
        f"بداية الصفحة {page_num+1}:\n{next_head}\n\n"
        "أجب بـ JSON فقط (لا تضف أي نص خارجه):\n"
        "{\"header\": نص الترويسة إن وجدت (عنوان كتاب أو فصل أو رقم صفحة) وإلا null,\n"
        " \"join\": true إذا كانت الجملة منقطعة بين الصفحتين وتستكمل في الصفحة التالية\n"
        "}"
    )

    try:
        msg = client.messages.create(
            model=model_name, max_tokens=80,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = msg.content[0].text.strip()
        # Extract JSON object even if the model adds surrounding text
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        data = json.loads(m.group()) if m else {}
    except Exception as e:
        print(f"  Boundary {page_num}→{page_num+1} skipped: {e}")
        return prev_content, next_content

    # 1. Strip the running header from the top of next_content
    header_text = data.get('header')
    if header_text:
        lines = next_content.splitlines()
        for idx, line in enumerate(lines):
            if line.strip() and (
                header_text.strip() in line or line.strip() in header_text
            ):
                lines[idx] = ''
                break
        next_content = '\n'.join(lines)

    # 2. Join split sentence across the boundary
    if data.get('join'):
        next_stripped = next_content.lstrip().splitlines()
        # Find first non-empty line in next page (after header removal)
        first_idx = next((i for i, l in enumerate(next_stripped) if l.strip()), None)
        if first_idx is not None:
            continuation = next_stripped[first_idx].lstrip()
            next_stripped[first_idx] = ''  # consumed into prev page
            prev_content = prev_content.rstrip() + ' ' + continuation
            next_content = '\n'.join(next_stripped)

    return prev_content, next_content


# --- Split corrected_pages into markers + content ---
markers, contents = [], []
for block in corrected_pages:
    lines = block.splitlines()
    if lines and re.match(r'--- Page \d+ ---', lines[0]):
        markers.append(lines[0])
        contents.append('\n'.join(lines[1:]))
    else:
        markers.append('')
        contents.append(block)

# --- Process every boundary ---
print(f'Stitching {len(contents)-1} page boundaries...')
for i in range(len(contents) - 1):
    print(f'  Page {i+1} → {i+2}...')
    contents[i], contents[i+1] = process_boundary(contents[i], contents[i+1], i+1)

# --- Rebuild page-marked output ---
stitched_pages = [
    f"{markers[i]}\n{contents[i]}".strip()
    for i in range(len(contents))
]

# --- Flowing text: remove page markers, collapse blank lines ---
flowing_text = re.sub(r'\n{3,}', '\n\n',
    '\n\n'.join(c.strip() for c in contents if c.strip())
).strip()

print('\nStitching complete.')
print('\n── Boundary preview (page 1 → 2) ──')
print(contents[0].strip()[-200:])
print('  ···  ')
print(contents[1].strip()[:200])

Stitching 2 page boundaries...
  Page 1 → 2...
  Page 2 → 3...

Stitching complete.

── Boundary preview (page 1 → 2) ──
 فإذا رضي عن صيغتها أخيراً عهد بها إلى عبد المسيح وزير رئيس شعبة الترجمة في وزارة الدفاع العراقية ليترجمها له إلى اللغة الإنكليزية ترجمة مبدئية، ويراجعها العسكري ويعاد طبعها على الآلة الكاتبة ثم يعاد.
  ···  
وعلى الرغم من أن جعفر العسكري قد وجد شيئاً من الاستقرار الذهني بعد قيام دولة العراق وتنصيب فيصل الأول ملكاً عليها، فإن حياته مع ذلك تخللها كثير من التنقلات بين بغداد ولندن وكثير من المشاغل والمشاكل ال


In [7]:
# Save both output formats
with open(os.path.join(output_dir, 'LLM_corrected_text.txt'), 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(stitched_pages))
print('Saved: LLM_corrected_text.txt  (page markers preserved)')

with open(os.path.join(output_dir, 'flowing_text.txt'), 'w', encoding='utf-8') as f:
    f.write(flowing_text)
print('Saved: flowing_text.txt         (clean flowing Arabic text)')

Saved: LLM_corrected_text.txt  (page markers preserved)
Saved: flowing_text.txt         (clean flowing Arabic text)


In [8]:
!zip -r /content/ara-ocr.zip /content/ara-ocr

  adding: content/ara-ocr/ (stored 0%)
  adding: content/ara-ocr/flowing_text.txt (deflated 68%)
  adding: content/ara-ocr/optimized_approach.txt (deflated 65%)
  adding: content/ara-ocr/LLM_corrected_text.txt (deflated 67%)
